### Step 1 - Read csv file

Take all environment varibles from below commnd

In [0]:
%run ../00-common/01.environment_config


In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
sources_path = f"{landing_folder_path}/{v_batch_id}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType,DateType

name_schema = StructType([
    StructField('givenName',StringType()),
    StructField('familyName',StringType())
    ])

schema=StructType([
    StructField('driverId',StringType()),

    StructField('name',name_schema),
    StructField('dateOfBirth',DateType()),
    StructField('nationality',StringType()),
    StructField('url',StringType())
    ]
)


In [0]:
drivers_df=(
    spark.read.format('json')
    #.option('header',True)
    #.option('inferSchema',True)
    .option('mode','FAILFAST')
    .schema(schema)
    .load(sources_path))

In [0]:
#display(circuit_df)

### Step 2 - Ingest time stamp and file path

In [0]:
from pyspark.sql.functions import *


In [0]:
drivers_df_final = add_ingestion_metadata(drivers_df)


### Step3- Write data to Bronze delta table

In [0]:
write_data_to_bronze(drivers_df_final,table_name, v_batch_id)

### Step4 - Read delta table using sql

In [0]:
 display(spark.table(table_name))